In [0]:
from pyspark.sql.functions import col, from_json, arrays_zip, explode, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType
#Reading bronze weather raw table
bronze_weather_df = spark.table("dev_project.flight_delay_lakehouse.bronze_weather_raw")
#Creating the weather schema
weather_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("timezone", StringType(), True),
    StructField("hourly", StructType([
        StructField("time", ArrayType(StringType()), True),
        StructField("temperature_2m", ArrayType(DoubleType()), True),
        StructField("precipitation", ArrayType(DoubleType()), True),
        StructField("wind_speed_10m", ArrayType(DoubleType()), True)
    ]), True)
])

#Parse the raw JSON string into a structured column
parsed_weather_df = bronze_weather_df.withColumn("weather_parsed", from_json(col("raw_response_json"), weather_schema))

#Zip the hourly arrays together so index positions stay aligned
zipped_weather_df = parsed_weather_df.withColumn("hourly_zipped", arrays_zip(
    col("weather_parsed.hourly.time"),
    col("weather_parsed.hourly.temperature_2m"),
    col("weather_parsed.hourly.precipitation"),
    col("weather_parsed.hourly.wind_speed_10m")
))

#Explode into one row per hour
exploded_weather_df = zipped_weather_df.withColumn("hourly_row", explode(col("hourly_zipped")))

#Select final Silver columns with clean names
silver_weather_df = exploded_weather_df.select(
    col("airport_iata"),
    to_timestamp(col("hourly_row.time")).alias("weather_ts"),
    col("hourly_row.temperature_2m").alias("air_temperature_2m_c"),
    col("hourly_row.precipitation").alias("precipitation_mm"),
    col("hourly_row.wind_speed_10m").alias("wind_speed_10m_kmh"),
    col("weather_kind"),
    col("ingested_at")
)

#Write Silver table
(silver_weather_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dev_project.flight_delay_lakehouse.silver_weather_hourly"))


In [0]:
%sql
SELECT * FROM dev_project.flight_delay_lakehouse.silver_weather_hourly